In [2]:
import os
import requests
import numpy as np
import pandas as pd

# Paths (this notebook lives in notebooks/, so ".." is the repo root)
RAW = "../data/raw"      # untouched downloads
OUT = "../data"          # final cleaned files
os.makedirs(RAW, exist_ok=True)

# Project constants
REF_YEAR = 2023          # reference year for HDI and all features
LOOKBACK = 2             # if 2023 is missing, also accept 2022 and 2021

In [3]:
import os
print(os.getcwd())

C:\Users\Admin\Documents\ds203-hdi-classification\notebooks


## 1. UNDP HDI table
- Source: UNDP Human Development Report 2025 release, *Composite indices complete time series*
- URL: https://hdr.undp.org/sites/default/files/2025_HDR/HDR25_Composite_indices_complete_time_series.csv
- Reference year: 2023 (column `hdi_2023`)
- Accessed: 23 Nov 2026

In [4]:
HDR_URL = ("https://hdr.undp.org/sites/default/files/2025_HDR/"
           "HDR25_Composite_indices_complete_time_series.csv")
hdr_path = os.path.join(RAW, "HDR25_composite_indices.csv")

if not os.path.exists(hdr_path):            # download only the first time
    r = requests.get(HDR_URL, timeout=120)
    r.raise_for_status()                    # stop with an error if the download failed
    with open(hdr_path, "wb") as f:
        f.write(r.content)
print("saved:", hdr_path, os.path.getsize(hdr_path) // 1024, "KB")

saved: ../data/raw\HDR25_composite_indices.csv 1954 KB


In [5]:
hdr = pd.read_csv(hdr_path, encoding="latin-1")
print(hdr.shape)
print(list(hdr.columns[:8]))
print("hdi_2023 present:", f"hdi_{REF_YEAR}" in hdr.columns)

(206, 1112)
['iso3', 'country', 'hdicode', 'region', 'hdi_rank_2023', 'hdi_1990', 'hdi_1991', 'hdi_1992']
hdi_2023 present: True


In [6]:
hdi = hdr[["iso3", "country", f"hdi_{REF_YEAR}"]].rename(columns={f"hdi_{REF_YEAR}": "hdi"})
hdi = hdi[~hdi["iso3"].str.startswith("ZZ")]        # drop regional/group aggregates
print("rows:", len(hdi))
print("missing HDI:", hdi["hdi"].isna().sum())
print(hdi[hdi["hdi"].isna()])
hdi.head()

rows: 195
missing HDI: 2
    iso3                              country  hdi
90   PRK  Korea (Democratic People's Rep. of)  NaN
115  MCO                               Monaco  NaN


,iso3,country,hdi
0,AFG,Afghanistan,0.496
1,ALB,Albania,0.810
2,DZA,Algeria,0.763
3,AND,Andorra,0.913
4,AGO,Angola,0.616


**Observation:** 195 countries after removing aggregates (iso3 starting with "ZZ").
2 countries have no 2023 HDI (DPR Korea, Monaco); they will be dropped because they have no target label.

## 2. World Bank WDI indicators
- Source: World Bank API (https://api.worldbank.org/v2), accessed 23 Nov 2026
- Year rule: use 2023; if missing, use the most recent value from 2021–2022; never a year after 2023.
  The year actually used is stored for every value.

In [9]:
INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG":    "inflation",
    "SL.UEM.TOTL.ZS":    "unemployment",
    "SP.DYN.IMRT.IN":    "infant_mortality",
    "EG.ELC.ACCS.ZS":    "electricity_access",
    "IT.NET.USER.ZS":    "internet_users",
    "NV.AGR.TOTL.ZS":    "agriculture_gdp",
    "NV.IND.TOTL.ZS":    "industry_gdp",
    "NE.TRD.GNFS.ZS":    "trade_gdp",
}

In [14]:
import json

def wb_get(endpoint, filename):
    """Call the World Bank API once, save the JSON in data/raw, and return the data rows."""
    path = os.path.join(RAW, filename)
    if not os.path.exists(path):
        url = f"https://api.worldbank.org/v2/{endpoint}&format=json&per_page=20000"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "w", encoding="utf-8") as f:
            f.write(r.text)
    with open(path, encoding="utf-8") as f:
        meta, rows = json.load(f)        # the API returns [page-info, data]
    return rows

first_year = REF_YEAR - LOOKBACK          # 2021
values, years = {}, {}

for code, name in INDICATORS.items():
    rows = wb_get(f"country/all/indicator/{code}?date={first_year}:{REF_YEAR}", f"wb_{code}.json")
    df = pd.DataFrame([{"iso3": r["countryiso3code"], "year": int(r["date"]), "value": r["value"]}
                       for r in rows if r["countryiso3code"]])
    df = df.dropna(subset=["value"]).sort_values("year")
    latest = df.groupby("iso3").last()     # most recent available year (<= 2023) per country
    values[name] = latest["value"]
    years[name] = latest["year"]
    print(f"{name:20s} countries with a value: {len(latest)}")

wdi = pd.DataFrame(values)        # one row per iso3, one column per indicator
wdi_year = pd.DataFrame(years)    # which year each value came from
wdi.head()

In [15]:
first_year = REF_YEAR - LOOKBACK          # 2021
values, years = {}, {}

for code, name in INDICATORS.items():
    rows = wb_get(f"country/all/indicator/{code}?date={first_year}:{REF_YEAR}", f"wb_{code}.json")
    df = pd.DataFrame([{"iso3": r["countryiso3code"], "year": int(r["date"]), "value": r["value"]}
                       for r in rows if r["countryiso3code"]])
    df = df.dropna(subset=["value"]).sort_values("year")
    latest = df.groupby("iso3").last()     # most recent available year (<= 2023) per country
    values[name] = latest["value"]
    years[name] = latest["year"]
    print(f"{name:20s} countries with a value: {len(latest)}")

wdi = pd.DataFrame(values)        # one row per iso3, one column per indicator
wdi_year = pd.DataFrame(years)    # which year each value came from
wdi.head()

gdp_growth           countries with a value: 254
inflation            countries with a value: 224
unemployment         countries with a value: 231
infant_mortality     countries with a value: 240
electricity_access   countries with a value: 259
internet_users       countries with a value: 213
agriculture_gdp      countries with a value: 241
industry_gdp         countries with a value: 244
trade_gdp            countries with a value: 222


,gdp_growth,inflation,unemployment,infant_mortality,electricity_access,internet_users,agriculture_gdp,industry_gdp,trade_gdp
iso3,,,,,,,,,
ABW,9.516360,NaN,NaN,NaN,100.000000,NaN,0.039529,10.837942,153.719158
AFE,1.941382,7.399186,7.676548,41.969085,50.667516,27.800000,15.305933,26.467975,56.283721
AFG,2.266944,-4.644709,14.008000,49.900000,85.300000,15.928400,34.743247,13.449823,66.999578
AFW,3.644591,5.221168,3.203355,57.353682,57.064737,38.500000,24.321381,23.244644,NaN
AGO,1.320166,13.644102,14.051000,33.000000,51.100000,37.885899,20.909337,35.701247,53.607013


## 3. Country metadata (region)
From the World Bank country API. Entries whose region is "Aggregates" (World, income groups, regional totals) are removed.

In [16]:
countries = wb_get("country?", "wb_countries.json")
meta = pd.DataFrame([{"iso3": c["id"],
                      "wb_name": c["name"],
                      "region": c["region"]["value"].strip()} for c in countries])
print("all entries:", len(meta))
meta = meta[meta["region"] != "Aggregates"]
print("real economies:", len(meta))
meta["region"].value_counts()

all entries: 296
real economies: 217


region
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3
Name: count, dtype: int64

## 4. Merge HDI + metadata + indicators on ISO3 codes

In [19]:
merged = meta.merge(hdi, on="iso3", how="outer", indicator=True)
only_wb  = merged[merged["_merge"] == "left_only"]
only_hdr = merged[merged["_merge"] == "right_only"]
print("in both:", (merged["_merge"] == "both").sum())
print("HDR countries with no World Bank match:", len(only_hdr), list(only_hdr["country"]))
print("World Bank economies not in HDR:", len(only_wb))

data = merged[merged["_merge"] == "both"].drop(columns=["_merge", "wb_name"])
data = data.merge(wdi, left_on="iso3", right_index=True, how="left")

n_before = len(data)
lost = data[data["hdi"].isna()]["country"].tolist()
data = data.dropna(subset=["hdi"]).reset_index(drop=True)
print(f"dropped for missing HDI: {n_before - len(data)} {lost}")
print("final countries:", len(data))

in both: 195
HDR countries with no World Bank match: 0 []
World Bank economies not in HDR: 22
dropped for missing HDI: 2 ['Monaco', "Korea (Democratic People's Rep. of)"]
final countries: 193


**Observation:** all 195 HDR countries matched a World Bank ISO3 code (no manual corrections needed).
22 World Bank economies (territories such as Puerto Rico and Greenland) have no HDI and are excluded.
Monaco and DPR Korea are dropped (no 2023 HDI). Final dataset: 193 countries.

## 5. Target and engineered feature

In [20]:
data["hdi_class"] = np.select([data["hdi"] < 0.550, data["hdi"] < 0.800], [0, 1], default=2)
counts = data["hdi_class"].value_counts().sort_index()
print(pd.DataFrame({"count": counts, "%": (counts / len(data) * 100).round(1)},
                   index=pd.Index([0, 1, 2], name="hdi_class")))

           count     %
hdi_class             
0             26  13.5
1             93  48.2
2             74  38.3


In [21]:
agr, ind = data["agriculture_gdp"], data["industry_gdp"]
serv = 100 - agr - ind
plausible = agr.notna() & ind.notna() & (agr >= 0) & (ind >= 0) & (serv >= 0)

shares = pd.DataFrame({"Agriculture": agr, "Industry": ind, "Services": serv})
data["DominantSector"] = shares.fillna(-1).idxmax(axis=1)   # name of the largest share
data.loc[~plausible, "DominantSector"] = np.nan             # missing / inconsistent → NaN
data["DominantSector"].value_counts(dropna=False)

DominantSector
Services       176
Industry         8
NaN              7
Agriculture      2
Name: count, dtype: int64

**Observation:** classes are imbalanced: Low 26 (13.5%), Medium/High 93 (48.2%), Very High 74 (38.3%).
DominantSector is almost always "Services" (176 of 186 known values); 8 Industry (mainly oil/mining economies),
2 Agriculture, 7 missing because a sector share is missing. It may be redundant with the numerical shares (as the brief notes).

## 6. Final dataset: counts, missingness, save

In [22]:
NUM = list(INDICATORS.values())
cols = ["iso3", "country", "region", "DominantSector"] + NUM + ["hdi", "hdi_class"]
data = data[cols].sort_values("iso3").reset_index(drop=True)

year_used = wdi_year.reindex(data["iso3"])               # year of every value, same row order
report = pd.DataFrame({
    "% missing": data[NUM + ["region", "DominantSector"]].isna().mean().mul(100).round(1),
    "values from 2021-2022": pd.Series({c: int((year_used[c] < REF_YEAR).sum()) for c in NUM}),
})
print("countries:", len(data))
report

countries: 193


,% missing,values from 2021-2022
DominantSector,3.6,NaN
agriculture_gdp,3.6,3.0
electricity_access,0.0,0.0
gdp_growth,2.1,1.0
industry_gdp,3.6,3.0
infant_mortality,1.0,0.0
inflation,7.8,3.0
internet_users,5.7,3.0
region,0.0,NaN
trade_gdp,11.9,3.0


In [23]:
data.to_csv(os.path.join(OUT, "hdi_wdi_2023.csv"), index=False)
year_used.reset_index().to_csv(os.path.join(OUT, "feature_year_used.csv"), index=False)
print("saved data/hdi_wdi_2023.csv and data/feature_year_used.csv")

saved data/hdi_wdi_2023.csv and data/feature_year_used.csv
